# HazeSignal: can fires and wind warn of haze earlier?

**Hypothesis:** more fire hotspots in Sumatra and Kalimantan, when winds carry air from those regions toward Kuala Lumpur, are associated with higher ground-level PM2.5 one or two days later.

This notebook uses September 2023 as a complete, reproducible pilot because the selected OpenAQ monitor has data for that month. September 2019 remains the intended severe-haze case study, but it cannot be tested honestly until daily DOE/APIMS PM2.5 observations are obtained. We keep the intermediate tables visible and treat a messy result as useful evidence, not something to hide.

## 1. Load the small analysis helpers

The notebook uses the same TypeScript functions as the command-line analysis. Run `npm run notebook` from the repository root. A Deno Jupyter kernel can also run the cells interactively when VS Code detects it, but it is optional.

In [ ]:
import { parse } from "npm:csv-parse@7.0.2/sync";
import { combineDailyData, dailyWind, initialBearing, isUsablePm25, KUALA_LUMPUR, linearRegression, malaysiaFireDate, scatterSvg, SOURCE_CENTROIDS, windTravelBearing, alignmentScore } from "../src/analysis.ts";
const readCsv = async <T>(path: string): Promise<T[]> => parse(await Deno.readTextFile(path), { columns: true, skip_empty_lines: true, trim: true }) as T[];

## 2. Check the source files

The repository includes the September 2023 CSV files produced by the three fetchers. API keys are never written into those files. Their dates and sources are recorded in `data/README.md`.

In [ ]:
const paths = {
  firms: "../data/firms_2023-09-01_2023-09-30.csv",
  wind: "../data/wind_2023-09-01_2023-09-30.csv",
  pm25: "../data/pm25_2023-09-01_2023-09-30.csv",
};

for (const [name, path] of Object.entries(paths)) {
  try { await Deno.stat(path); console.log(`✓ ${name}: ${path}`); }
  catch { console.log(`✗ ${name}: ${path} is missing`); }
}

## 3. Fire hotspots and fire intensity

The FIRMS file contains one row per VIIRS heat detection, not one row per separate fire. FIRMS timestamps are UTC, so we convert them to Malaysia time (UTC+8) before grouping detections by day.

We test both hotspot count and fire radiative power (FRP). FRP estimates radiant heat release in megawatts. It is a useful intensity clue, but it does not directly measure how much PM2.5 a fire emits.

In [ ]:
const hotspots = await readCsv<Record<string, string>>(paths.firms);
const fireCounts = Object.groupBy(hotspots, row => malaysiaFireDate({
  acq_date: row.acq_date,
  acq_time: row.acq_time,
  frp: row.frp,
  region: row.region as "sumatra" | "kalimantan",
}));
console.table(Object.entries(fireCounts).slice(0, 10).map(([date, rows]) => ({
  date,
  hotspot_count: rows?.length ?? 0,
  fire_radiative_power_mw: rows?.reduce((sum, row) => sum + Number(row.frp), 0).toFixed(1),
})));

## 4. Wind direction and the source-to-Malaysia bearing

A compass bearing is calculated from each source centre to Kuala Lumpur. If `lat₁, lon₁` is the source and `lat₂, lon₂` is Kuala Lumpur, the initial bearing is:

`atan2(sin(Δlon) cos(lat₂), cos(lat₁) sin(lat₂) − sin(lat₁) cos(lat₂) cos(Δlon))`

Meteorological direction says where wind comes **from**, so smoke travel is `(wind direction + 180°) mod 360°`. The alignment score is `max(0, cos(travel bearing − route bearing))`. This makes the vector calculation visible instead of hiding it in a weather library.

In [ ]:
const routes = Object.entries(SOURCE_CENTROIDS).map(([region, source]) => ({
  region,
  bearing_to_kl: initialBearing(source.latitude, source.longitude, KUALA_LUMPUR.latitude, KUALA_LUMPUR.longitude),
}));
console.table(routes);

const workedExample = { wind_from: 225, smoke_travels_toward: windTravelBearing(225) };
console.log(workedExample, "Sumatra alignment:", alignmentScore(225, routes[0].bearing_to_kl));

In [ ]:
const rawWind = await readCsv<Record<string, string>>(paths.wind);
const wind = dailyWind(rawWind.map(row => ({
  date: row.date,
  wind_speed_kmh: Number(row.wind_speed_kmh),
  wind_direction_degrees: Number(row.wind_direction_degrees),
})));
console.table(wind.slice(0, 10));

## 5. Ground-level PM2.5 and data quality

This pilot uses daily OpenAQ PM2.5 from sensor 2085316 in Kuala Lumpur. PM2.5 is the mass of particles no wider than about 2.5 micrometres in one cubic metre of air. We use µg/m³, or micrograms per cubic metre.

A daily value is used only when at least 75% of the day is covered. The monitor is missing 27 September, and 28 September has only 29% coverage, so that incomplete target is excluded rather than treated as a normal observation.

In [ ]:
const rawPm25 = await readCsv<Record<string, string>>(paths.pm25);
const pm25 = rawPm25.map(row => ({
  date: row.date,
  pm25_ug_m3: Number(row.pm25_ug_m3),
  coverage_percent: Number(row.coverage_percent),
}));
console.table(pm25);

## 6. Combine each date with later PM2.5

For every wind date, we calculate four simple source clues: hotspot count, wind-aligned hotspot count, total FRP and wind-aligned FRP. We then look up PM2.5 one and two calendar days later.

A hotspot contributes 1 to the aligned count when the simplified wind route points directly toward Kuala Lumpur and 0 when it points sideways or away. Intermediate columns stay visible so the reasoning can be checked.

In [ ]:
const combined = combineDailyData(hotspots.map(row => ({
  acq_date: row.acq_date,
  acq_time: row.acq_time,
  frp: row.frp,
  region: row.region as "sumatra" | "kalimantan",
})), wind, pm25);
console.table(combined.map(row => ({
  date: row.date,
  hotspots: row.hotspot_count,
  aligned_hotspots: row.aligned_hotspot_count.toFixed(1),
  fire_power_mw: row.fire_radiative_power_mw.toFixed(1),
  aligned_fire_power_mw: row.aligned_fire_radiative_power_mw.toFixed(1),
  pm25: row.pm25_ug_m3,
  pm25_next_day: row.pm25_next_day,
  next_day_coverage: row.pm25_next_day_coverage_percent,
})));

## 7. Fit the simple next-day lines

We fit next-day PM2.5 = intercept + slope × predictor. Pearson's r describes how closely two quantities rise together along a straight line inside this sample. It is not an accuracy percentage.

The comparison between raw and wind-aligned predictors tests whether wind adds useful information. The FRP comparison tests whether estimated heat release works better than treating every hotspot equally.

In [ ]:
const reliableNextDay = combined.filter(row =>
  isUsablePm25(row.pm25_next_day, row.pm25_next_day_coverage_percent)
);
const predictors = [
  { name: "hotspot count", values: reliableNextDay.map(row => row.hotspot_count) },
  { name: "wind-aligned hotspots", values: reliableNextDay.map(row => row.aligned_hotspot_count) },
  { name: "fire intensity", values: reliableNextDay.map(row => row.fire_radiative_power_mw) },
  { name: "wind-aligned intensity", values: reliableNextDay.map(row => row.aligned_fire_radiative_power_mw) },
];
const nextDayResults = predictors.map(predictor => {
  const points = predictor.values.map((x, index) => ({ x, y: reliableNextDay[index].pm25_next_day! }));
  return { predictor: predictor.name, points, regression: linearRegression(points) };
});
console.table(nextDayResults.map(result => ({
  predictor: result.predictor,
  r: result.regression.r,
  r_squared: result.regression.rSquared,
  observations: result.regression.observations,
})));
const alignedResult = nextDayResults[1];
const svg = scatterSvg(alignedResult.points, alignedResult.regression);
await Deno.writeTextFile("../data/regression.svg", svg);
display({ "image/svg+xml": svg }, { raw: true });

## 8. Interpret the result honestly

After removing the low-coverage target, September 2023 contains 27 usable next-day observations. Hotspot count has r = 0.552, while wind-aligned hotspot count has r = 0.696. The September relationship looks positive, but it is measured on the same dates used to fit the line.

The stronger check is npm run validate. It learns from September–October and predicts 54 sufficiently complete target days in November–December that were kept out of fitting. Hotspots alone average 2.80 µg/m³ error, almost tied with the 2.85 baseline. Hotspots plus wind average 5.91; FRP averages 3.12; FRP plus wind averages 6.15.

The wind-aligned and intensity models perform worse on unseen dates. The current method therefore does not yet support a reliable early-warning claim. This is useful evidence: local 10 m wind, regional centres and fire heat release are too simple to represent plume transport and smoke production.

In [ ]:
const reliableTwoDay = combined.filter(row =>
  isUsablePm25(row.pm25_in_two_days, row.pm25_in_two_days_coverage_percent)
);
console.table([
  ...nextDayResults.slice(0, 2).map(result => ({
    predictor: result.predictor,
    lead: "1 day",
    r: result.regression.r,
    r_squared: result.regression.rSquared,
    observations: result.regression.observations,
  })),
  {
    predictor: "hotspot count",
    lead: "2 days",
    ...linearRegression(reliableTwoDay.map(row => ({ x: row.hotspot_count, y: row.pm25_in_two_days! }))),
  },
  {
    predictor: "wind-aligned hotspots",
    lead: "2 days",
    ...linearRegression(reliableTwoDay.map(row => ({ x: row.aligned_hotspot_count, y: row.pm25_in_two_days! }))),
  },
]);

## 9. What a fuller model needs

The physical chain is fire material → combustion products → atmospheric transport → measured particles. A better model needs rainfall, humidity, plume height, winds sampled along the route, exact fire locations, fire age, several haze and non-haze seasons, and multiple Malaysian monitors.

The live command uses the result as a cautious clue rather than a forecast:

```powershell
npm start
```

For the complete explanation and admissions-ready discussion, read RESEARCH_REPORT.md.